working out how to scrape draft comps

In [16]:
import requests
import re
import bs4 as bs

In [1]:
url= "https://www.nbadraft.net/actual-draft/?year-mock=2022"

In [4]:
r = requests.get(url)

In [7]:
html_data = r.content

## get lists of players by year, and urls

In [9]:
soup = bs.BeautifulSoup(html_data, 'html.parser')

In [12]:
table_soup = soup.find(id='nba-mock-draft-content')

In [19]:
table_soup

<div class="nba-mock-draft-content oh vc_row-fluid vc_row-flex" id="nba-mock-draft-content"><div class="vc_row"><div class="table-responsive col-1 vc_col-xs-12 vc_col-lg-6 col-6" id="nba_consensus_mock1"><table class="nba_mock_consensus_table mock-draft-table sof-tablepress sticky-enabled"><thead><tr><th class="rank">#</th><th class="team">Team</th><th class="player">Player</th><th class="height">H</th><th class="weight">W</th><th class="teamposition">P</th><th class="school">School</th><th class="class">C</th></tr></thead><tbody><tr class="odd"><td class="rank">1</td><td class="team"><noscript><img class="team-logo" src="https://nbadraft.net/wp-content/uploads/2010/06/orl_25_0.gif"/></noscript> <span style="font-weight:bold;">Orlando</span></td><td class="player"><a href="https://www.nbadraft.net/players/paolo-banchero/">Paolo Banchero</a></td><td class="height">6-10</td><td class="weight">250</td><td class="teamposition">PF/C</td><td class="school">Duke</td><td class="class">Fr.</td>

In [20]:
# this is added by js..
#table_columns = table_soup.find_all(class_=re.compile("dataTables_wrapper"))



In [21]:
round_one = soup.find(id="nba_consensus_mock1")

In [27]:
# first row is a header

rowz = round_one.find_all("tr")[1:]

In [ ]:
table_columns = ['order', 'team', 'player_name', 'full_url', 'height', 'weight', 'position', 'alma_mater', 'school_class']

In [ ]:
players = []

for row in rowz:
    cells = row.find_all('td')

    player_data = {}

    for cell in cells:
        attr = cell.attrs['class'][0]

        if attr == 'team':
            ## need to strip non-ascii in team names
            player_data[attr] = cell.text.encode('ascii', 'ignore').decode().replace('*', '')
        else:
            player_data[attr] = cell.text
        # pickup url to full player comp
        if attr == 'player':
            player_data['player_url'] = cell.find('a')['href']


    players.append(player_data)

In [87]:
import pandas as pd

In [88]:
foo = pd.DataFrame(players)

In [65]:
cells[0].attrs['class'][0]

'rank'

### scraping individual player page


In [92]:
player_url = "https://www.nbadraft.net/players/dexter-pittman/"

r = requests.get(player_url)

In [94]:
soup = bs.BeautifulSoup(r.content, 'html.parser')
analysis = soup.find(id='analysis')

In [142]:
player = {}
descr_strengths = []
descr_weaknesses = []
descr_other = []


paras = analysis.find(class_='vc_tta-panel-body').find_all('p')

for p in paras:
    if 'NBA Comparison:' in p.text:
        comp_chunks = p.text.split('NBA Comparison:')
        player['nbadraft_net_comp'] = comp_chunks[1]

    if len(p.text) < 100:
        pass
    else:
        #print(f"on {p.text}")

        if 'Strengths' in p.text:
            if 'Weaknesses' in p.text:
                strength, weakness = p.text.split('Weaknesses')
                descr_strengths.append(strength[len('Strengths:'):].strip())
                descr_weaknesses.append(weakness[1:].strip())
            else:
                descr_strengths.append(p.text[len('Strengths:'):].strip())

        elif 'Weaknesses' in p.text:
            descr_weaknesses.append(p.text[len('Weaknesses:'):].strip())


        else:
            descr_other.append(p.text)

player['descr_other'] = descr_other
player['descr_strengths'] = descr_strengths
player['descr_weaknesses'] = descr_weaknesses
player['descr_raw'] = analysis


In [141]:
analysis

<div class="vc_tta-panel vc_active" data-vc-content=".vc_tta-panel-body" id="analysis"><div class="vc_tta-panel-heading"><h4 class="vc_tta-panel-title"><a data-vc-accordion="" data-vc-container=".vc_tta-container" href="#analysis"><span class="vc_tta-title-text">Analysis</span></a></h4></div><div class="vc_tta-panel-body"><div class="wpb_text_column wpb_content_element"><div class="wpb_wrapper"><p><span class="text"><b>NBA Comparison: Stanley Roberts</b></span></p><p><strong>Strengths: </strong>A physically imposing low post player … Has continued to work on his body, losing a lot of weight and improving his conditioning … Very efficient scorer … He has a very strong and wide frame and he is able to create room and muscle his way around the hoop … Operating on the low block he is able to establish terrific position and is also very capable of making tough catches in traffic … He uses his strength to force his way to the hoop, overpowering defenders for easy finishes inside … Has a nice

In [143]:
player

{'nbadraft_net_comp': ' Stanley Roberts',
 'descr_other': [],
 'descr_strengths': ['A physically imposing low post player … Has continued to work on his body, losing a lot of weight and improving his conditioning … Very efficient scorer … He has a very strong and wide frame and he is able to create room and muscle his way around the hoop … Operating on the low block he is able to establish terrific position and is also very capable of making tough catches in traffic … He uses his strength to force his way to the hoop, overpowering defenders for easy finishes inside … Has a nice mini hook going to his left shoulder … A good athlete for his size, he has the explosiveness to catch and finish strong at the rim … Extremely effective rebounder on the offensive end, he clears space and attacks the glass … Shows good timing and is able to block a lot of shots in the paint …',
  'Mammoth of a man who has made enormous strides developing his game. Has dropped over 70 pounds since arriving on the

In [130]:
descr_strengths

['A physically imposing low post player … Has continued to work on his body, losing a lot of weight and improving his conditioning … Very efficient scorer … He has a very strong and wide frame and he is able to create room and muscle his way around the hoop … Operating on the low block he is able to establish terrific position and is also very capable of making tough catches in traffic … He uses his strength to force his way to the hoop, overpowering defenders for easy finishes inside … Has a nice mini hook going to his left shoulder … A good athlete for his size, he has the explosiveness to catch and finish strong at the rim … Extremely effective rebounder on the offensive end, he clears space and attacks the glass … Shows good timing and is able to block a lot of shots in the paint …',
 'Mammoth of a man who has made enormous strides developing his game. Has dropped over 70 pounds since arriving on the UT campus, from 366 to the 290 range — showing his work ethic and desire to make u

In [131]:
descr_weaknesses

['Has no ability to step out and shoot jumpers or put it on the floor, which limits him strictly to the center position and at 6’10 he is a bit undersized … Even with his improved physique, his motor is still very much a question mark … He has problems staying in the game for long stretches … Extremely foul prone, he is fairly undisciplined and doesn’t move his feet … Does not operate well with or finish to his left hand … Extremely poor foul shooter …',
 'Stamina is a major concern. Still only seeing 15 minutes of playing time per contest … Despite loss of weight, movement up and down the court is still an issue and can be painful at times … He will always be a ‘lumberer’ … Foul trouble usually emerges as a result of fatigue … He averages a foul every 5 minutes on the court … He has made massive improvements in lateral quickness and agility, but these will always be flaws … Has a tendency to push off on the offensive and defensive glass, committing unnecessary fouls. Always enjoys a m

In [106]:
player

{'nbadraft_net_comp': ' Stanley Roberts',
 'descr': ['Strengths: A physically imposing low post player … Has continued to work on his body, losing a lot of weight and improving his conditioning … Very efficient scorer … He has a very strong and wide frame and he is able to create room and muscle his way around the hoop … Operating on the low block he is able to establish terrific position and is also very capable of making tough catches in traffic … He uses his strength to force his way to the hoop, overpowering defenders for easy finishes inside … Has a nice mini hook going to his left shoulder … A good athlete for his size, he has the explosiveness to catch and finish strong at the rim … Extremely effective rebounder on the offensive end, he clears space and attacks the glass … Shows good timing and is able to block a lot of shots in the paint …   Weaknesses: Has no ability to step out and shoot jumpers or put it on the floor, which limits him strictly to the center position and at 6

In [102]:
paras[0].text

'NBA Comparison: Stanley Roberts'

In [148]:
analysis

<div class="vc_tta-panel vc_active" data-vc-content=".vc_tta-panel-body" id="analysis"><div class="vc_tta-panel-heading"><h4 class="vc_tta-panel-title"><a data-vc-accordion="" data-vc-container=".vc_tta-container" href="#analysis"><span class="vc_tta-title-text">Analysis</span></a></h4></div><div class="vc_tta-panel-body"><div class="wpb_text_column wpb_content_element"><div class="wpb_wrapper"><p><span class="text"><b>NBA Comparison: Stanley Roberts</b></span></p><p><strong>Strengths: </strong>A physically imposing low post player … Has continued to work on his body, losing a lot of weight and improving his conditioning … Very efficient scorer … He has a very strong and wide frame and he is able to create room and muscle his way around the hoop … Operating on the low block he is able to establish terrific position and is also very capable of making tough catches in traffic … He uses his strength to force his way to the hoop, overpowering defenders for easy finishes inside … Has a nice

In [182]:
q = {'r': 17}
j = {'g': 12}

q.update(j)
q

{'r': 17, 'g': 12}

In [190]:
def scrape_player_page(player_url):
    soup = fetch_player_page(player_url)
    prose = extract_prose(soup)
    numerics = extract_numerics(soup)
    prose.update(numerics)
    return prose

def fetch_player_page(player_url):
    r = requests.get(player_url)
    soup = bs.BeautifulSoup(r.content, 'html.parser')
    return soup

def extract_numerics(soup):
    attrs = soup.find(class_="player-attributes").find_all(class_ = 'div-table-row')
    parsed_attrs = {}
    for attr in attrs:
        attr_name = attr.find(class_='attribute-name').text.strip()
        attr_value = attr.find(class_='attribute-value').text.strip()
        parsed_attrs[attr_name] = attr_value
    return parsed_attrs

def extract_prose(soup):
    analysis = soup.find(id='analysis')

    player = {}
    descr_strengths = []
    descr_weaknesses = []
    descr_other = []

    paras = analysis.find(class_='vc_tta-panel-body').find_all(['p', 'h3'])

    got_comp = False
    ## sometimes comp is not in a paragraph

    for p in paras:
        if 'NBA Comparison:' in p.text:
            comp_chunks = p.text.split('NBA Comparison:')
            player['nbadraft_net_comp'] = comp_chunks[1].strip()
            continue

        if len(p.text) < 100:
            pass
        else:
            if 'Strengths' in p.text:
                if 'Weaknesses' in p.text:
                    strength, weakness = p.text.split('Weaknesses')
                    descr_strengths.append(strength[len('Strengths:'):].strip())
                    descr_weaknesses.append(weakness[1:].strip())
                else:
                    descr_strengths.append(p.text[len('Strengths:'):].strip())

            elif 'Weaknesses' in p.text:
                descr_weaknesses.append(p.text[len('Weaknesses:'):].strip())
            else:
                descr_other.append(p.text)

    player['descr_other'] = descr_other
    player['descr_strengths'] = descr_strengths
    player['descr_weaknesses'] = descr_weaknesses
    player['descr_raw'] = analysis

    return player


In [191]:
scrape_player_page("https://www.nbadraft.net/players/nikola-jokic/")

{'nbadraft_net_comp': 'Nikola Vucivic',
 'descr_other': [],
 'descr_strengths': ['Very high basketball IQ, This is his greatest strength. Strong personality. A team player. Has a great work ethic … Tremendous length … Can really shoot the ball … Hard worker … He was first introduced to real professional training – strength and conditioning as well as basketball workouts in September of 2013 and has shown tremendous improvement in his body … Possesses a big wingpan … Considering the fact that he competes in the Adriatic league, where most players are older than he is, and also that he’s an average athlete, he uses his basketball IQ to succeed… Nikola understands the game really well. Furthermore he is able to follow and execute coach’s instructions. Shows a winning attitude, competes hard in games and practices … A self starter, doesn’t need to constantly be pushed to get the most of his abilties … Well liked by teammates, outgoing, strong character, doesn’t drink or smoke … Young playe

In [192]:
nik = fetch_player_page("https://www.nbadraft.net/players/nikola-jokic/")

In [196]:
nik.find(class_='attribute-big-board').find(class_='value').text

'61'

In [197]:
def extract_ranks(soup):
    ranks = {}
    ranks['mock'] = soup.find(class_='attribute-mock').find(class_='value').text
    ranks['big_board'] = soup.find(class_='attribute-big-board').find(class_='value').text
    ranks['overall'] = soup.find(class_='attribute-overall').find(class_='value').text

    return ranks

In [198]:
extract_ranks(nik)

{'mock': '53', 'big_board': '61', 'overall': '85'}

In [193]:
extract_numerics(nik)

{'Athleticism': '6',
 'Size': '9',
 'Defense': '7',
 'Strength': '7',
 'Quickness': '6',
 'Leadership': '7',
 'Jump Shot': '8',
 'NBA Ready': '7',
 'Rebounding': '7',
 'Potential': '7',
 'Post Skills': '7',
 'Intangibles': '7'}

In [189]:
extract_prose(nik)

{'nbadraft_net_comp': 'Nikola Vucivic',
 'descr_other': [],
 'descr_strengths': ['Very high basketball IQ, This is his greatest strength. Strong personality. A team player. Has a great work ethic … Tremendous length … Can really shoot the ball … Hard worker … He was first introduced to real professional training – strength and conditioning as well as basketball workouts in September of 2013 and has shown tremendous improvement in his body … Possesses a big wingpan … Considering the fact that he competes in the Adriatic league, where most players are older than he is, and also that he’s an average athlete, he uses his basketball IQ to succeed… Nikola understands the game really well. Furthermore he is able to follow and execute coach’s instructions. Shows a winning attitude, competes hard in games and practices … A self starter, doesn’t need to constantly be pushed to get the most of his abilties … Well liked by teammates, outgoing, strong character, doesn’t drink or smoke … Young playe

In [179]:
parsed_attrs = {}

attrs = nik.find(class_="player-attributes").find_all(class_ = 'div-table-row')
for attr in attrs:
    #print(attr)
    attr_name = attr.find(class_='attribute-name').text.strip()
    attr_value = attr.find(class_='attribute-value').text.strip()
    parsed_attrs[attr_name] = attr_value

parsed_attrs

{'Athleticism': '6',
 'Size': '9',
 'Defense': '7',
 'Strength': '7',
 'Quickness': '6',
 'Leadership': '7',
 'Jump Shot': '8',
 'NBA Ready': '7',
 'Rebounding': '7',
 'Potential': '7',
 'Post Skills': '7',
 'Intangibles': '7'}

In [176]:
attr.find(class_="div-table-cell attribute-name")

<div class="div-table-cell attribute-name"> Athleticism</div>

In [166]:
nik.find(class_="player-attributes").find_all(class_='div-table-row')

[<div class="div-table-row"><div class="div-table-cell attribute-name"> Athleticism</div><div class="div-table-cell attribute-boxes"><div class="boxes"><div class="box"></div><div class="box"></div><div class="box"></div><div class="box"></div><div class="box"></div><div class="box"></div></div></div><div class="div-table-cell attribute-value"> 6</div></div>,
 <div class="div-table-row"><div class="div-table-cell attribute-name"> Size</div><div class="div-table-cell attribute-boxes"><div class="boxes"><div class="box"></div><div class="box"></div><div class="box"></div><div class="box"></div><div class="box"></div><div class="box"></div><div class="box"></div><div class="box"></div><div class="box"></div></div></div><div class="div-table-cell attribute-value"> 9</div></div>,
 <div class="div-table-row"><div class="div-table-cell attribute-name"> Defense</div><div class="div-table-cell attribute-boxes"><div class="boxes"><div class="box"></div><div class="box"></div><div class="box"></d

In [155]:
nik.find(id='analysis').find(class_='vc_tta-panel-body').find_all(['p', 'h3'])

[<h3>NBA Comparison: Nikola Vucivic</h3>,
 <p><strong>Strengths: </strong>Very high basketball IQ, This is his greatest strength. Strong personality. A team player. Has a great work ethic … Tremendous length … Can really shoot the ball … Hard worker … He was first introduced to real professional training – strength and conditioning as well as basketball workouts in September of 2013 and has shown tremendous improvement in his body … Possesses a big wingpan … Considering the fact that he competes in the Adriatic league, where most players are older than he is, and also that he’s an average athlete, he uses his basketball IQ to succeed… Nikola understands the game really well. Furthermore he is able to follow and execute coach’s instructions. Shows a winning attitude, competes hard in games and practices … A self starter, doesn’t need to constantly be pushed to get the most of his abilties … Well liked by teammates, outgoing, strong character, doesn’t drink or smoke … Young player, born 

In [146]:
import pprint

In [147]:
pprint.pprint(scrape_player_page("https://www.nbadraft.net/players/nikola-jokic/"))

{'descr_other': [],
 'descr_raw': <div class="vc_tta-panel vc_active" data-vc-content=".vc_tta-panel-body" id="analysis"><div class="vc_tta-panel-heading"><h4 class="vc_tta-panel-title"><a data-vc-accordion="" data-vc-container=".vc_tta-container" href="#analysis"><span class="vc_tta-title-text">Analysis</span></a></h4></div><div class="vc_tta-panel-body"><div class="wpb_text_column wpb_content_element"><div class="wpb_wrapper"><h3>NBA Comparison: Nikola Vucivic</h3><p><strong>Strengths: </strong>Very high basketball IQ, This is his greatest strength. Strong personality. A team player. Has a great work ethic … Tremendous length … Can really shoot the ball … Hard worker … He was first introduced to real professional training – strength and conditioning as well as basketball workouts in September of 2013 and has shown tremendous improvement in his body … Possesses a big wingpan … Considering the fact that he competes in the Adriatic league, where most players are older than he is, and al

In [199]:
bag

NameError: name 'df' is not defined

In [203]:
type(analysis)

bs4.element.Tag

In [202]:
analysis.html

In [204]:
str(analysis)

'<div class="vc_tta-panel vc_active" data-vc-content=".vc_tta-panel-body" id="analysis"><div class="vc_tta-panel-heading"><h4 class="vc_tta-panel-title"><a data-vc-accordion="" data-vc-container=".vc_tta-container" href="#analysis"><span class="vc_tta-title-text">Analysis</span></a></h4></div><div class="vc_tta-panel-body"><div class="wpb_text_column wpb_content_element"><div class="wpb_wrapper"><p><span class="text"><b>NBA Comparison: Stanley Roberts</b></span></p><p><strong>Strengths: </strong>A physically imposing low post player … Has continued to work on his body, losing a lot of weight and improving his conditioning … Very efficient scorer … He has a very strong and wide frame and he is able to create room and muscle his way around the hoop … Operating on the low block he is able to establish terrific position and is also very capable of making tough catches in traffic … He uses his strength to force his way to the hoop, overpowering defenders for easy finishes inside … Has a nic

fixme: Ty lawson isn't getting Strengths section even though he has them.